In [1]:
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms
atoms = MagresAtoms.load_magres('/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/8-HQ-ipc2-B_opt_magres_new.magres')

In [2]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [3]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [4]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [5]:
for atom in atoms.species('N'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

14N1 sigma:
 [[  54.66251289 -170.99997289  -41.7198033 ]
 [-176.10106493  -10.02487149   65.24874007]
 [ -50.1724841    59.39702504 -101.14156868]]

14N2 sigma:
 [[  54.66251289  170.99997289  -41.7198033 ]
 [ 176.10106493  -10.02487149  -65.24874007]
 [ -50.1724841   -59.39702504 -101.14156868]]

14N3 sigma:
 [[  54.66251289 -170.99997289  -41.7198033 ]
 [-176.10106493  -10.02487149   65.24874007]
 [ -50.1724841    59.39702504 -101.14156868]]

14N4 sigma:
 [[  54.66251289  170.99997289  -41.7198033 ]
 [ 176.10106493  -10.02487149  -65.24874007]
 [ -50.1724841   -59.39702504 -101.14156868]]



In [7]:
for atom in atoms.species('N'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

14N1 sigma:
 -1.7924882422171076

14N2 sigma:
 -1.7924882422171475

14N3 sigma:
 -1.7924882422171

14N4 sigma:
 -1.792488242217114



In [17]:
# using values from latest magres file
efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)
efg[0,0]= 0.1397; efg[0,1]= -0.0113; efg[0,2]= -0.1522;
efg[1,0]= efg[0,1]; efg[1,1]= 0.1884; efg[1,2]= 0.0017;
efg[2,0]= efg[0,2]; efg[2,1]= efg[1,2]; efg[2,2]= -0.3281;
efg[:,:] = atoms.species('N')[0].efg.V
print(efg)

[[ 0.13971379 -0.01133996 -0.15217568]
 [-0.01133996  0.18836131  0.00167133]
 [-0.15217568  0.00167133 -0.3280751 ]]


In [ ]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

CS_total[:,:] = atoms.species('N').ms.sigma[0]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species('N')[0].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf
Q = 0.0204 # You need to change this value for different nucleus [value is for 14N in barn]
V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[ 0.67  -0.054 -0.729]
 [-0.054  0.903  0.008]
 [-0.729  0.008 -1.573]]

CS Tensor:
 [[  54.663 -171.     -41.72 ]
 [-176.101  -10.025   65.249]
 [ -50.172   59.397 -101.142]]

CS isotropic Tensor:
 [[-18.835   0.      0.   ]
 [  0.    -18.835   0.   ]
 [  0.      0.    -18.835]]

CS symmetric Tensor:
 [[  54.663 -173.551  -45.946]
 [-173.551  -10.025   62.323]
 [ -45.946   62.323 -101.142]]

CS antisymmetric Tensor:
 [[ 0.     2.551  4.226]
 [-2.551  0.     2.926]
 [-4.226 -2.926  0.   ]]


In [14]:
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

 Unsorted Eigenvalues:
 [-1.78897982  0.83945593  0.9495239 ] 

 Unsorted Eigenvectors:
 [[ 0.28447805  0.72725937  0.62463273]
 [ 0.00289128  0.65089944 -0.75915846]
 [ 0.95867819 -0.2177699  -0.18306389]] 

Sorted Eigenvalues: 
 [ 0.83945593  0.9495239  -1.78897982] 

Sorted Eigenvectors: 
 [[ 0.72725937  0.62463273  0.28447805]
 [ 0.65089944 -0.75915846  0.00289128]
 [-0.2177699  -0.18306389  0.95867819]] 


 Unsorted Eigenvalues:
 [ 216.68714994 -161.87445122 -111.316626  ] 

 Unsorted Eigenvectors:
 [[-0.74095795  0.52811893  0.41481527]
 [ 0.63065229  0.75948539  0.15956071]
 [ 0.23077911 -0.37983198  0.89580616]] 

Sorted Eigenvalues: 
 [-111.316626   -161.87445122  216.68714994] 

Sorted Eigenvectors: 
 [[ 0.41481527  0.52811893 -0.74095795]
 [ 0.15956071  0.75948539  0.63065229]
 [ 0.89580616 -0.37983198  0.23077911]] 



In [15]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 0.8394559276392304 0.9495238968644022 -1.7889798245036488
CSA Tensor Components δyy, δxx, δzz: 
 -111.3166259963974 -161.87445122198875 216.6871499388843


In [16]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]
print(tabulate(table, headers=['Qauntity', 'Value']))

Qauntity            Value
------------  -----------
CQ (MHz)       -1.78898
etaq            0.0615256
iso_cs (ppm)  -18.8346
csa (ppm)     235.522
etas            0.214663


In [12]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[ 0.62383996  0.72792843  0.28450642]
 [-0.75984922  0.65009323  0.0028223 ]
 [-0.18290126 -0.21794264  0.95866998]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
49.996039688974676 16.530183661685534 -0.5683547891552087 

Direction cosine csa: 

[[ 0.52811893  0.41481527 -0.74095795]
 [ 0.75948539  0.15956071  0.63065229]
 [-0.37983198  0.89580616  0.23077911]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
-67.02250436578022 76.65705436089141 40.402136749309506 



In [13]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: -77.77492183631688 chi: 89.30019460025116 xi: -10.351796062797385 



**Rotation of tensors Crystal--> Tenon Frame**

In [14]:
#Euler angles Crystal--> Tenon Frame for LHQ
alpha = 280
beta = 72.5
gamma = 180
U  = Rabc(alpha, beta, gamma)
Cs_tenon = np.matmul(np.matmul(np.linalg.inv(U), Cs), U)

V_tenon = np.matmul(np.matmul(np.linalg.inv(U), V), U)

print('CSA Tensor in Crystal Frame: \n', Cs) #from magres
print('CSA Tensor in Tenon Frame: \n', Cs_tenon)

print('==========================')
print('Quad Tensor in Crystal Frame: \n', V)  #from magres
print('Quad Tensor in Tenon Frame: \n', V_tenon)


CSA Tensor in Crystal Frame: 
 [[  54.66251289 -173.55051891  -45.9461437 ]
 [-173.55051891  -10.02487149   62.32288255]
 [ -45.9461437    62.32288255 -101.14156868]]
CSA Tensor in Tenon Frame: 
 [[-49.73125225 113.56003833 130.25268606]
 [113.56003833 -20.99313773 106.55662867]
 [130.25268606 106.55662867  14.2204627 ]]
Quad Tensor in Crystal Frame: 
 [[ 0.6696212  -0.05416406 -0.7295372 ]
 [-0.05416406  0.90305393  0.00814858]
 [-0.7295372   0.00814858 -1.57267513]]
Quad Tensor in Tenon Frame: 
 [[ 0.83890449  0.33975064 -0.1669801 ]
 [ 0.33975064 -0.88732299  1.23036223]
 [-0.1669801   1.23036223  0.0484185 ]]
